# Validación CrowdFeatureExtractor

Valida el módulo de Crowd Features (MDLF, Zhao et al. 2022) para cercanías.

**Antes de ejecutar necesitas:**
1. `x_pred_checked` — salida de `check_transbordo` con columnas `tipo`, `tipo_transporte`, `prediccion_descenso`, `timestamp_validacion`, `id_tarjeta`, `punto_acceso`.
2. `x_test`, `y_test` — salida de `crear_mapeos` (pares subida-descenso REAL).

**Checks del notebook:**
1. Tamaño del conjunto UNLINKED cercanías con descenso real
2. Ajuste del extractor (`fit`) — índice agregado
3. Shape de salida
4. Sanidad proporciones: suma de F_c^m por viaje ≈ 1 cuando hay señal
5. Sanidad normalización per-station (norma L2 ≈ 1)
6. Coherencia temporal: semana 1 (más reciente) debe tener más señal que semana 2
7. Distribución de señal por semana
8. Verificación cruzada con historial: la estación más visitada colectivamente coincide con la de mayor señal crowd
9. Guardado del output

In [ ]:
import sys
import importlib.util
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Importar CrowdFeatureExtractor (directorio con espacio → importlib)
_CROWD_FILE = Path('.') / 'crowd_features.py'
spec = importlib.util.spec_from_file_location('crowd_features', _CROWD_FILE)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
CrowdFeatureExtractor = mod.CrowdFeatureExtractor

# station_index desde el LDA ya entrenado
THETA_CERCANIAS = Path('../LDA/output/theta_cercanias.parquet')
theta_cercanias = pd.read_parquet(THETA_CERCANIAS)
station_index   = theta_cercanias.index.tolist()
N = len(station_index)
print(f'Estaciones cercanías (N): {N}')
print(f'Primeras 5: {station_index[:5]}')

---
## 0 — Carga de datos de entrada

Adaptar las rutas a los ficheros reales del pipeline.

In [ ]:
# ─── ADAPTAR ESTAS RUTAS ────────────────────────────────────────────
# x_pred_checked: salida de check_transbordo con columna 'tipo'
# x_pred_checked = pd.read_parquet('ruta/a/x_pred_checked.parquet')

# x_test, y_test: salida de crear_mapeos
# x_test = pd.read_parquet('ruta/a/x_test.parquet')
# y_test = pd.read_parquet('ruta/a/y_test.parquet')
# ────────────────────────────────────────────────────────────────────

print('Recuerda asignar: x_pred_checked, x_test, y_test')

---
## 1 — Tamaño del conjunto UNLINKED cercanías con descenso real

In [ ]:
# Viajes UNLINKED de cercanías
unlinked_c = x_pred_checked[
    (x_pred_checked['tipo'] == 'UNLINKED') &
    (x_pred_checked['tipo_transporte'] == 'cercanias')
].copy()
print(f'UNLINKED cercanías total:              {len(unlinked_c):>10,}')

# Intersección con crear_mapeos → viajes con etiqueta real
merge_keys = ['id_tarjeta', 'timestamp_validacion']
labeled_c = unlinked_c.merge(
    x_test[merge_keys].drop_duplicates(), on=merge_keys, how='inner'
)
labeled_c = labeled_c.merge(
    y_test[merge_keys + ['punto_acceso']].rename(columns={'punto_acceso': 'destino_real'}),
    on=merge_keys, how='left'
)
labeled_c = labeled_c[labeled_c['destino_real'].notna()].reset_index(drop=True)

print(f'UNLINKED cercanías con descenso REAL:  {len(labeled_c):>10,}  ← conjunto supervisado')
print(f'Porcentaje sobre UNLINKED:             {len(labeled_c)/len(unlinked_c)*100:>9.1f}%')
print(f'Destinos únicos (estaciones bajada):   {labeled_c["destino_real"].nunique():>10,}')
print()

# ¿Cuántos viajes de x_pred_checked tienen tipo cercanías? (para crowd)
cercanias_all = x_pred_checked[x_pred_checked['tipo_transporte'] == 'cercanias']
print(f'Total viajes cercanías en x_pred_checked (todos tipos): {len(cercanias_all):,}')
print(f'  de los cuales con prediccion_descenso válida (no unknown): '
      f"{(cercanias_all['prediccion_descenso'] != 'unknown').sum():,}")

In [ ]:
# Distribución de destinos reales en el labeled set
top20 = labeled_c['destino_real'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(top20)), top20.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Top 20 estaciones destino reales en labeled_c')
ax.set_ylabel('Nº viajes UNLINKED con ese destino real')
plt.tight_layout()
plt.show()

in_si = labeled_c['destino_real'].isin(station_index).mean() * 100
print(f'Destinos reales en station_index: {in_si:.1f}%')

---
## 2 — Ajuste del extractor (fit)

`fit` recibe `x_pred_checked` completo — todos los usuarios, todos los modos.
Internamente filtra solo cercanías y pre-agrega por `(origen, franja, fecha)`.

In [ ]:
N_WEEKS = 2   # semanas de historial crowd (mismo que PipelineMDLF)

ext = CrowdFeatureExtractor(
    red='04',
    station_index=station_index,
    n_weeks=N_WEEKS,
    norm='l2',
)
print(ext)

t0 = time.time()
ext.fit(x_pred_checked)   # TODOS los usuarios, filtra cercanías internamente
dt = time.time() - t0
print(f'\nfit completado en {dt:.1f}s')
print(f'Claves (origen, franja) en el índice: {len(ext._agg):,}')
print(f'Ejemplo de claves (primeras 5):')
for k in list(ext._agg.keys())[:5]:
    df_k = ext._agg[k]
    print(f'  {k}  →  {len(df_k):,} registros | '
          f'fechas [{df_k["fecha"].min()} … {df_k["fecha"].max()}]')

In [ ]:
# Cobertura: ¿cuántas de las N estaciones tienen al menos una clave (origen, *) en el índice?
origenes_en_indice = {k[0] for k in ext._agg.keys()}
cobertura = len(origenes_en_indice & set(station_index)) / N * 100
print(f'Estaciones cercanías con datos OD en el índice: {cobertura:.1f}% de las {N}')

# Franjas únicas en el índice
franjas_en_indice = {k[1] for k in ext._agg.keys()}
print(f'Franjas únicas en el índice: {sorted(franjas_en_indice)}')

---
## 3 — Shape de salida

In [ ]:
# Transform sobre muestra pequeña primero
sample = labeled_c.head(200).copy()

t0 = time.time()
features_sample = ext.transform(sample)
print(f'transform ({len(sample)} filas) en {time.time()-t0:.2f}s')
print(f'Shape de salida: {features_sample.shape}')
print(f'Esperado:        ({len(sample)}, {N * N_WEEKS + 1})')
print(f'Columnas (primeras 6): {features_sample.columns[:6].tolist()}')
print(f'Columnas (últimas 3):  {features_sample.columns[-3:].tolist()}')

In [ ]:
# CHECK shape
assert features_sample.shape == (len(sample), N * N_WEEKS + 1), \
    f'Shape incorrecto: {features_sample.shape}'
print('✓ Shape correcto')

# CHECK columnas: deben seguir el patrón {si}__w{m}
feat_cols = [c for c in features_sample.columns if c != 'has_crowd_signal']
assert len(feat_cols) == N * N_WEEKS
ejemplo_col = feat_cols[0]
assert '__w' in ejemplo_col, f'Columna inesperada: {ejemplo_col}'
print(f'✓ Nombres de columnas correctos. Ejemplo: {ejemplo_col}')

# CHECK has_crowd_signal ∈ {0, 1}
assert features_sample['has_crowd_signal'].isin([0.0, 1.0]).all()
n_signal = int(features_sample['has_crowd_signal'].sum())
print(f'✓ has_crowd_signal: {n_signal} viajes con señal, {len(sample)-n_signal} sin señal')

---
## 4 — Sanidad proporciones: suma de F_c^m ≈ 1 cuando hay señal

Cada vector F_c^m es una distribución de proporciones: debe sumar ≈ 1 si hay viajes en esa ventana.

In [ ]:
for m in range(1, N_WEEKS + 1):
    w_cols = [c for c in feat_cols if c.endswith(f'__w{m}')]
    row_sums = features_sample[w_cols].sum(axis=1)

    # Solo filas donde esta semana tiene señal
    has_signal_m = row_sums > 0
    n_with = has_signal_m.sum()

    print(f'Semana w{m}:')
    print(f'  Viajes con señal: {n_with} de {len(sample)}')
    if n_with > 0:
        sumas = row_sums[has_signal_m]
        print(f'  Suma media de proporciones: {sumas.mean():.4f}  (debe ser ≈ 1.0)')
        print(f'  Min / Max: {sumas.min():.4f} / {sumas.max():.4f}')
        assert abs(sumas.mean() - 1.0) < 0.01, \
            f'Suma de proporciones incorrecta en w{m}: {sumas.mean():.4f}'
        print(f'  ✓ Proporciones correctas')

---
## 5 — Sanidad normalización per-station (norma L2 ≈ 1)

Cada fila de la matriz N×n_weeks se normaliza con L2.
Para filas con al menos un valor no-nulo, la norma debe ser ≈ 1.

In [ ]:
mat = features_sample[feat_cols].values.reshape(len(sample), N, N_WEEKS)  # (n, N, W)
norms = np.linalg.norm(mat, axis=2)   # (n, N) — norma L2 de cada fila por estación

# Solo viajes con señal y estaciones con al menos un valor no-nulo
signal_mask = features_sample['has_crowd_signal'].values == 1
if signal_mask.any():
    norms_signal = norms[signal_mask]            # (n_signal, N)
    norms_nonzero = norms_signal[norms_signal > 0]  # excluir estaciones con todo cero
    print(f'Filas con señal: {signal_mask.sum()}')
    print(f'Norma L2 media per-station (estaciones no-nulo): {norms_nonzero.mean():.4f}')
    print(f'Std: {norms_nonzero.std():.6f}')
    assert abs(norms_nonzero.mean() - 1.0) < 0.01, \
        f'Normalización incorrecta: media norma = {norms_nonzero.mean():.4f}'
    print('✓ Normalización per-station correcta')
else:
    print('ℹ Sin filas con señal en la muestra — no se puede verificar normalización')

---
## 6 — Coherencia temporal: w1 (más reciente) debe tener más señal que w2

In [ ]:
signal_por_semana = {}
for m in range(1, N_WEEKS + 1):
    w_cols = [c for c in feat_cols if c.endswith(f'__w{m}')]
    signal_por_semana[f'w{m}'] = (features_sample[w_cols].sum(axis=1) > 0).mean() * 100

print('Porcentaje de viajes con señal por semana:')
for sem, pct in signal_por_semana.items():
    bar = '█' * int(pct / 2)
    print(f'  {sem}: {pct:5.1f}%  {bar}')

# w1 debe tener >= señal que w2 (datos más recientes deberían ser más densos o iguales)
if N_WEEKS >= 2:
    if signal_por_semana['w1'] >= signal_por_semana['w2']:
        print('✓ w1 tiene >= señal que w2 (coherente: más reciente = más cubierto)')
    else:
        print('ℹ w2 tiene más señal que w1 — puede ocurrir si el período de datos'
              ' cubre más la semana anterior que la más reciente')

In [ ]:
# Visualizar distribución de señal por semana
fig, axes = plt.subplots(1, N_WEEKS, figsize=(6 * N_WEEKS, 4), sharey=False)
if N_WEEKS == 1:
    axes = [axes]

for m, ax in enumerate(axes, start=1):
    w_cols = [c for c in feat_cols if c.endswith(f'__w{m}')]
    row_sums = features_sample[w_cols].sum(axis=1)
    ax.hist(row_sums[row_sums > 0], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(f'w{m} — distribución de suma de proporciones\n(viajes con señal)')
    ax.set_xlabel('Suma proporciones (≈ 1 si una estación domina)')
    ax.set_ylabel('Nº viajes')

plt.tight_layout()
plt.show()

---
## 7 — Distribución de señal crowd por estación

¿Qué estaciones concentran más demanda colectiva en el conjunto de muestra?

In [ ]:
# Para w1: suma de proporciones por estación sobre todos los viajes
# → estaciones que más aparecen como destino colectivo en la semana más reciente
w1_cols = [c for c in feat_cols if c.endswith('__w1')]
station_signal_w1 = features_sample[w1_cols].sum(axis=0)
station_signal_w1.index = [c.replace('__w1', '') for c in station_signal_w1.index]
station_signal_w1 = station_signal_w1.sort_values(ascending=False)

top15 = station_signal_w1.head(15)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(top15)), top15.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(top15)))
ax.set_xticklabels(top15.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Top 15 estaciones por señal crowd w1 (muestra 200 viajes)')
ax.set_ylabel('Suma de proporciones crowd sobre todos los viajes')
plt.tight_layout()
plt.show()

print('Top 5 estaciones con mayor señal crowd (w1):')
for s, v in top15.head(5).items():
    print(f'  {s}: {v:.3f}')

---
## 8 — Verificación cruzada: señal crowd vs. destinos reales del labeled set

Las estaciones con mayor señal crowd deben solapar con las más frecuentes como destino real.
No será perfecto (crowd = todos los usuarios, no solo UNLINKED), pero el solapamiento debe ser considerable.

In [ ]:
# Top estaciones crowd vs top estaciones destino real
top_crowd = set(station_signal_w1.head(15).index)
top_real  = set(labeled_c['destino_real'].value_counts().head(15).index)

overlap = top_crowd & top_real
print(f'Top 15 crowd: {sorted(top_crowd)}')
print()
print(f'Top 15 destinos reales: {sorted(top_real)}')
print()
print(f'Solapamiento: {len(overlap)} de 15 estaciones')
print(f'  En común: {sorted(overlap)}')

if len(overlap) >= 5:
    print('✓ Solapamiento razonable entre señal crowd y destinos reales')
else:
    print('ℹ Bajo solapamiento — puede ser normal si crowd y UNLINKED tienen patrones distintos')

In [ ]:
# Verificación manual para una estación concreta
# Para un viaje con señal, la estación con mayor proporción crowd debería ser plausible
if features_sample['has_crowd_signal'].any():
    idx_ejemplo = features_sample[features_sample['has_crowd_signal'] == 1].index[0]
    viaje = sample.loc[idx_ejemplo]

    print(f'Viaje ejemplo (idx={idx_ejemplo}):')
    print(f'  Origen: {viaje["punto_acceso"]}')
    print(f'  Timestamp: {viaje["timestamp_validacion"]}')
    print(f'  Destino real: {sample.loc[idx_ejemplo, "destino_real"] if "destino_real" in sample.columns else "N/A"}')

    # Estaciones con mayor señal crowd para este viaje (w1)
    row_w1 = features_sample.loc[idx_ejemplo, w1_cols]
    row_w1.index = [c.replace('__w1', '') for c in row_w1.index]
    top5_este_viaje = row_w1.sort_values(ascending=False).head(5)
    print(f'\n  Top 5 estaciones por señal crowd w1 para este viaje:')
    for s, v in top5_este_viaje.items():
        marker = ' ← destino real' if 'destino_real' in sample.columns and s == sample.loc[idx_ejemplo, 'destino_real'] else ''
        print(f'    {s}: {v:.4f}{marker}')

---
## 9 — Transform completo y guardado

In [ ]:
t0 = time.time()
features_full = ext.transform(labeled_c)
print(f'transform completo ({len(labeled_c):,} viajes) en {time.time()-t0:.1f}s')
print(f'Shape: {features_full.shape}')
print(f'Memoria: {features_full.memory_usage(deep=True).sum() / 1e6:.1f} MB')

# has_crowd_signal en el conjunto completo
pct_signal = features_full['has_crowd_signal'].mean() * 100
print(f'\n% viajes con señal crowd: {pct_signal:.1f}%')

# Señal por semana en el conjunto completo
print('\nSeñal por semana (conjunto completo):')
for m in range(1, N_WEEKS + 1):
    w_cols_m = [c for c in features_full.columns if c.endswith(f'__w{m}')]
    pct_m = (features_full[w_cols_m].sum(axis=1) > 0).mean() * 100
    print(f'  w{m}: {pct_m:.1f}% de viajes con señal')

In [ ]:
# Añadir metadatos para facilitar uso posterior
features_full['destino_real'] = labeled_c['destino_real'].values
features_full['id_tarjeta']   = labeled_c['id_tarjeta'].values
features_full['punto_acceso'] = labeled_c['punto_acceso'].values

# Checks finales
assert features_full.shape[0] == len(labeled_c)
assert features_full.shape[1] == N * N_WEEKS + 1 + 3   # features + flag + 3 metadatos
print('✓ Shape final correcto')

In [ ]:
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

out_path = out_dir / 'crowd_cercanias.parquet'
features_full.to_parquet(out_path)
print(f'Guardado en: {out_path.resolve()}')
print(f'Shape: {features_full.shape}')